# Acting on layers after they exist

One targeting vocabulary — an id or name, `types=`, `exclude_types=`, `group=` —
shared by everything that operates on existing layers: `hide`, `show`, `select`,
`highlight`, `find_layers`, `bounds_of`. Learn it once.

This notebook builds one map and then drives it from cells below. Keep the map in
view while you run them.

In [ ]:
import numpy as np
import pandas as pd
from swiftmap import Map

rng = np.random.default_rng(9)
n = 80
sites = pd.DataFrame({
    "lat": 36.05 + rng.normal(0, 0.05, n),
    "lon": -5.45 + rng.normal(0, 0.08, n),
    "site": [f"S{i:03d}" for i in range(n)],
    "reading": np.round(rng.gamma(4, 4, n), 1),
})
steps = 30
tracks = pd.DataFrame({
    "track_id": np.repeat(["Vessel A", "Vessel B"], steps),
    "step": np.tile(np.arange(steps), 2),
    "lat": np.concatenate([36.00 + np.cumsum(rng.normal(0.002, 0.003, steps)),
                           35.96 + np.cumsum(rng.normal(0.003, 0.003, steps))]),
    "lon": np.concatenate([-5.75 + np.cumsum(rng.normal(0.010, 0.005, steps)),
                           -5.70 + np.cumsum(rng.normal(0.008, 0.005, steps))]),
})
zones = pd.DataFrame({
    "zone_id": ["North"] * 4 + ["South"] * 4,
    "vertex": [0, 1, 2, 3] * 2,
    "lat": [36.10, 36.10, 36.16, 36.16, 35.95, 35.95, 36.01, 36.01],
    "lon": [-5.55, -5.40, -5.40, -5.55, -5.55, -5.40, -5.40, -5.55],
})

m = Map()
m.add_circle_markers(sites, name="Sites", layer_group="Sensors")
m.add_line(tracks, line_id_col="track_id", order_col="step",
           name="track_id", layer_group="Tracks", weight=3)
m.add_polygon(zones, shape_id_col="zone_id", order_col="vertex",
              name="zone_id", layer_group="Zones", fill_opacity=0.25)
m

## Finding layers

`find_layers` takes the whole vocabulary and returns the matches; `get_layer`
returns one by id, name, or `(group, name)`. Collection children share a name by
design, so `types=` is what tells them apart.

In [ ]:
[(l.get("name"), l.get("type")) for l in m.find_layers(group="Zones")]

In [ ]:
[(l.get("name"), l.get("type")) for l in m.find_layers(types="polyline")]

## Hide and show

A call that matches **nothing warns** — a mistyped name looks identical to a
hidden layer, and silence would bury the mistake.

In [ ]:
m.hide(group="Zones")
m.show("North");            # back on, by name

In [ ]:
import warnings
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    m.hide("Not A Layer")
[str(w.message) for w in caught]

## Select: declarative and total

Each call states the *complete* selection for its scope — everything else in scope
hides, and `select(None, scope=...)` restores a clean slate. Nothing about the
previous selection is remembered, which is what keeps repeated calls from
drifting. Criteria select on their own, exactly as they do in `hide`:
`m.select(group="Zones")`, `m.select(types="polyline")`.

In [ ]:
m.select(["Vessel A"], scope="Tracks", zoom=True, zoom_offset=-1);

In [ ]:
m.select(None, scope="Tracks");     # clean slate for that folder

In [ ]:
m.select(types="polygon", scope="Zones");   # criteria alone — no target needed

## Highlight: mark without disturbing

Highlights sit in a field of their own **above** the layer's styling — clearing
restores what was underneath with nothing remembered. Per-family overrides let a
mixed selection read right: an accent on points, a wash on areas.

In [ ]:
m.highlight(["North", "Vessel B"], color="#ffcc00",
            lines={"weight": 6}, polygons={"fill_opacity": 0.5});

In [ ]:
m.highlight(None);                  # everything back to its own style

## One feature, by index

`set_feature_styles` overrides individual features inside a layer — a hovered row,
a table selection. Same contract: each call replaces the last, `{}` clears.

In [ ]:
m.set_feature_styles("Sites", {3: {"color": "#ffcc00", "radius": 14},
                               17: {"color": "#e15759", "radius": 14}});

In [ ]:
m.set_feature_styles("Sites", {});

## Updating, batching, removing

`update_layer` rewrites attributes on a layer. Several mutations leave as one
message inside `with m.batch():` — every `add_*` already batches internally, so
this is for grouping calls. Removal cleans up the layer's coordinate buffers too.

In [ ]:
with m.batch():
    m.update_layer("Vessel A", color="orange")
    m.hide(group="Zones")
    m.fit_bounds(m.bounds_of(["Vessel A", "Vessel B"]), padding=40)

In [ ]:
m.remove_layers(m.find_layers(types="polyline"))
len(m.layers)

## Updating a layer's data in place

`update_layer(data=...)` swaps what a layer draws while the layer itself stays
put — same id, name, folder, visibility, time animation and highlight. That is
what makes a refresh survivable: the sidebar checkbox a user just unticked is
still unticked afterwards. `append=True` grows a point layer instead, and sends
only the new rows.

In [ ]:
m.hide("Sites")                                  # a user's choice, mid-session
m.update_layer("Sites", data=sites.sample(40, random_state=1))
print("still hidden after the swap:", m.get_layer("Sites").get("visible") is False)

In [ ]:
fresh = pd.DataFrame({"lat": 36.05 + rng.normal(0, 0.05, 25),
                      "lon": -5.45 + rng.normal(0, 0.08, 25),
                      "site": [f"N{i:03d}" for i in range(25)],
                      "reading": np.round(rng.gamma(4, 4, 25), 1)})
m.update_layer("Sites", data=fresh, append=True)
m.show("Sites");

## Clicks come back to Python

Every click reports — features and open map alike. A feature click sets
`m.clicked_layer_id` and `m.selected_index` (overlaps resolved top-down: points
over lines over polygons); a click on open map clears them and reports
`m.clicked_latlng` as `[lat, lon]`. Feature clicks record their location too — a
point reports its own coordinates, not the mouse's. `m.click_seq` bumps on
**every** click, so one observer reads "where" and "on what" from a single event.

Reacting from Shiny is `shiny/02_linked_table.py`'s whole loop, and drawing on
the map — AOIs coming back to Python as GeoJSON — is **12_draw_aoi**.